In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sobhanmoosavi/us-accidents/US_Accidents_March23.csv


In [2]:
import pandas as pd

path = '/kaggle/input/datasets/sobhanmoosavi/us-accidents/US_Accidents_March23.csv'
df = pd.read_csv(path, low_memory=False)
print(df.shape)
df.head()

(7728394, 46)


,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,A-1,Source2,3,2016-02-08 05:46:00,2016-02-08 11:00:00,39.865147,-84.058723,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Night
1,A-2,Source2,2,2016-02-08 06:07:59,2016-02-08 06:37:59,39.928059,-82.831184,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Day
2,A-3,Source2,2,2016-02-08 06:49:27,2016-02-08 07:19:27,39.063148,-84.032608,NaN,NaN,0.01,...,False,False,False,False,True,False,Night,Night,Day,Day
3,A-4,Source2,3,2016-02-08 07:23:34,2016-02-08 07:53:34,39.747753,-84.205582,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Day,Day,Day
4,A-5,Source2,2,2016-02-08 07:39:07,2016-02-08 08:09:07,39.627781,-84.188354,NaN,NaN,0.01,...,False,False,False,False,True,False,Day,Day,Day,Day


In [3]:
import numpy as np

DROP_COLS = ["Country", "End_Lat", "End_Lng", "Wind_Chill(F)", "Description", "Turning_Loop"]

def clean(df):
    df.drop(columns=DROP_COLS, inplace=True, errors="ignore")
    df.loc[~df["Temperature(F)"].between(-80, 140), "Temperature(F)"] = np.nan
    df.loc[df["Wind_Speed(mph)"] > 200, "Wind_Speed(mph)"] = np.nan
    df.loc[df["Visibility(mi)"] > 20, "Visibility(mi)"] = np.nan
    df.loc[~df["Pressure(in)"].between(25, 35), "Pressure(in)"] = np.nan

    df["Precipitation(in)"] = df["Precipitation(in)"].fillna(0)
    num_cols = df.select_dtypes(include="number").columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].fillna("Unknown")

    df["Start_Time"] = pd.to_datetime(df["Start_Time"], errors="coerce")
    df["End_Time"] = pd.to_datetime(df["End_Time"], errors="coerce")
    bool_cols = df.select_dtypes(include="bool").columns
    df[bool_cols] = df[bool_cols].astype(np.int8)

    df.dropna(subset=["Start_Time", "End_Time"], inplace=True)
    return df

df_clean = clean(df.copy())
print(df_clean.shape)

(6985228, 40)


### Output

Cleaned the dataset by removing low-value columns, fixing physically impossible sensor 
readings (extreme temperature, wind speed, visibility, pressure), and dropping rows with 
unparseable timestamps.

- Shape before cleaning: (7,728,394, 46)
- Shape after cleaning: (6,985,228, 40)
- Rows removed: 743,166 (mostly bad timestamps)
- Columns removed: 6 (Country, End_Lat, End_Lng, Wind_Chill(F), Description, Turning_Loop)

## Feature Engineering

In [4]:
def engineer_features(df):
    df["start_year"] = df["Start_Time"].dt.year
    df["start_month"] = df["Start_Time"].dt.month
    df["start_day"] = df["Start_Time"].dt.day
    df["start_hour"] = df["Start_Time"].dt.hour
    df["start_minute"] = df["Start_Time"].dt.minute
    df["start_dayofweek"] = df["Start_Time"].dt.dayofweek
    df["is_weekend"] = df["start_dayofweek"].isin([5, 6]).astype(np.int8)

    df["is_morning_rush"] = df["start_hour"].between(7, 9).astype(np.int8)
    df["is_evening_rush"] = df["start_hour"].between(16, 18).astype(np.int8)
    df["is_rush_hour"] = (df["is_morning_rush"] | df["is_evening_rush"]).astype(np.int8)

    df["duration_min"] = (df["End_Time"] - df["Start_Time"]).dt.total_seconds() / 60
    df.loc[df["duration_min"] < 0, "duration_min"] = np.nan
    p99 = df["duration_min"].quantile(0.99)
    print(f"99th percentile duration: {p99:.1f} min ({p99/60:.1f} hrs)")
    df = df[df["duration_min"].between(0, p99)].copy()

    df["hour_sin"] = np.sin(2 * np.pi * df["start_hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["start_hour"] / 24)
    df["dayofweek_sin"] = np.sin(2 * np.pi * df["start_dayofweek"] / 7)
    df["dayofweek_cos"] = np.cos(2 * np.pi * df["start_dayofweek"] / 7)
    df["month_sin"] = np.sin(2 * np.pi * df["start_month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["start_month"] / 12)
    return df

df_features = engineer_features(df_clean.copy())
print(df_features.shape)

99th percentile duration: 779.9 min (13.0 hrs)
(6915377, 57)


### Output

Added calendar features (year, month, day, hour, weekday), rush-hour and weekend flags, 
trip duration (capped at 99th percentile = 779.9 min / 13.0 hrs to remove outliers), and 
cyclical encodings (sin/cos) for hour, day-of-week, and month.

- Shape before feature engineering: (6,985,228, 40)
- Shape after feature engineering: (6,915,377, 57)
- Rows removed: 69,851 (duration outliers beyond 99th percentile)
- Columns added: 17

##  Feature selection + train/test split 

In [5]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

DROP_FOR_MODEL = [
    "ID", "Start_Time", "End_Time", "Weather_Timestamp",
    "Street", "Zipcode", "Airport_Code", "City", "County",
    "start_hour", "start_dayofweek", "start_month",
    "is_morning_rush", "is_evening_rush",
    "start_day", "start_minute",
    "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight",
]
drop_cols = [c for c in DROP_FOR_MODEL if c in df_features.columns]

X = df_features.drop(columns=drop_cols + ["Severity"])
y = df_features["Severity"]

for col in X.select_dtypes(include=["object", "category"]).columns:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# Two sample sizes — LightGBM/RF use 500k, LogReg/MLP use 100k (matches original notebook)
X_500k, _, y_500k, _ = train_test_split(X, y, train_size=500_000, random_state=42, stratify=y)
X_100k, _, y_100k, _ = train_test_split(X, y, train_size=100_000, random_state=42, stratify=y)

X_train_lg, X_test_lg, y_train_lg, y_test_lg = train_test_split(
    X_500k, y_500k, test_size=0.2, random_state=42, stratify=y_500k
)
X_train_sm, X_test_sm, y_train_sm, y_test_sm = train_test_split(
    X_100k, y_100k, test_size=0.2, random_state=42, stratify=y_100k
)

scaler = StandardScaler()
X_train_sm_sc = scaler.fit_transform(X_train_sm)
X_test_sm_sc = scaler.transform(X_test_sm)

print(f"Features: {X.shape[1]}")
print("Splits ready")

Features: 37
Splits ready


## Install MLflow + LightGBM 

In [6]:
!pip install mlflow lightgbm -q
import mlflow
print(mlflow.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 81.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 72.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 42.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.9/214.9 kB 12.0 MB/s eta 0:00:00
3.14.0


## Logistic Regression + MLflow


In [7]:
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import time

mlflow.set_experiment("accident-severity-models")

with mlflow.start_run(run_name="logistic_regression"):
    t0 = time.time()

    params = {"C": 1, "solver": "lbfgs", "max_iter": 500}
    lr = LogisticRegression(class_weight="balanced", random_state=42, n_jobs=-1, **params)
    lr.fit(X_train_sm_sc, y_train_sm)

    y_pred = lr.predict(X_test_sm_sc)
    f1 = f1_score(y_test_sm, y_pred, average="weighted")
    train_time = time.time() - t0

    mlflow.log_params(params)
    mlflow.log_metric("weighted_f1", f1)
    mlflow.log_metric("train_time_sec", train_time)
    mlflow.sklearn.log_model(lr, "model")

    print(f"Logistic Regression — F1: {f1:.4f} | Time: {train_time:.1f}s")

2026/06/23 06:12:12 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/23 06:12:12 INFO mlflow.store.db.utils: Updating database tables
2026/06/23 06:12:15 INFO mlflow.tracking.fluent: Experiment with name 'accident-severity-models' does not exist. Creating a new experiment.
2026/06/23 06:12:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Logistic Regression — F1: 0.5179 | Time: 3.0s


## Random Forest + MLflow

In [8]:
from sklearn.ensemble import RandomForestClassifier

with mlflow.start_run(run_name="random_forest"):
    t0 = time.time()

    params = {"n_estimators": 200, "max_depth": 15}
    rf = RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1, **params)
    rf.fit(X_train_lg, y_train_lg)

    y_pred = rf.predict(X_test_lg)
    f1 = f1_score(y_test_lg, y_pred, average="weighted")
    train_time = time.time() - t0

    mlflow.log_params(params)
    mlflow.log_metric("weighted_f1", f1)
    mlflow.log_metric("train_time_sec", train_time)
    mlflow.sklearn.log_model(rf, "model")

    print(f"Random Forest — F1: {f1:.4f} | Time: {train_time:.1f}s")

2026/06/23 06:14:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Random Forest — F1: 0.7273 | Time: 90.0s


##  LightGBM + MLflow

In [9]:
import lightgbm as lgb

with mlflow.start_run(run_name="lightgbm"):
    t0 = time.time()

    params = {"n_estimators": 200, "max_depth": 6, "learning_rate": 0.1, "num_leaves": 63}
    lgbm = lgb.LGBMClassifier(class_weight="balanced", random_state=42, n_jobs=-1, verbose=-1, **params)
    lgbm.fit(X_train_lg, y_train_lg)

    y_pred = lgbm.predict(X_test_lg)
    f1 = f1_score(y_test_lg, y_pred, average="weighted")
    train_time = time.time() - t0

    mlflow.log_params(params)
    mlflow.log_metric("weighted_f1", f1)
    mlflow.log_metric("train_time_sec", train_time)
    mlflow.lightgbm.log_model(lgbm, "model")

    print(f"LightGBM — F1: {f1:.4f} | Time: {train_time:.1f}s")

2026/06/23 06:15:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


LightGBM — F1: 0.7601 | Time: 27.3s


## MLP Neural Network + MLflow

In [10]:
from sklearn.neural_network import MLPClassifier
from sklearn.utils import resample

with mlflow.start_run(run_name="mlp_balanced"):
    t0 = time.time()

    print("Balancing training data for MLP...")
    train_data = pd.concat([X_train_sm.copy(), y_train_sm.rename("Severity")], axis=1)
    max_size = train_data["Severity"].value_counts().max()

    upsampled = []
    for sev in train_data["Severity"].unique():
        subset = train_data[train_data["Severity"] == sev]
        upsampled.append(resample(subset, replace=True, n_samples=max_size, random_state=42))

    train_balanced = pd.concat(upsampled)
    X_train_mlp = scaler.transform(train_balanced.drop(columns="Severity"))
    y_train_mlp = train_balanced["Severity"]
    print(f"Balanced shape: {X_train_mlp.shape}")

    params = {
        "hidden_layer_sizes": (128, 64),
        "activation": "relu",
        "learning_rate_init": 0.001,
        "max_iter": 30,
        "early_stopping": True,
        "validation_fraction": 0.1,
    }
    mlp = MLPClassifier(random_state=42, **params)
    mlp.fit(X_train_mlp, y_train_mlp)

    y_pred = mlp.predict(X_test_sm_sc)
    f1 = f1_score(y_test_sm, y_pred, average="weighted")
    train_time = time.time() - t0

    mlflow.log_params(params)
    mlflow.log_metric("weighted_f1", f1)
    mlflow.log_metric("train_time_sec", train_time)
    mlflow.sklearn.log_model(mlp, "model", serialization_format="cloudpickle")

    print(f"MLP (balanced) — F1: {f1:.4f} | Time: {train_time:.1f}s")

Balancing training data for MLP...
Balanced shape: (248444, 37)


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(
2026/06/23 06:16:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/23 06:16:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


MLP (balanced) — F1: 0.7353 | Time: 75.9s


In [11]:
print(f"F1 (full precision): {f1:.10f}")

F1 (full precision): 0.7353289099


In [12]:
print("Before balancing:")
print(y_train_sm.value_counts())
print("\nAfter balancing:")
print(y_train_mlp.value_counts())


Before balancing:
Severity
2    62111
3    15023
4     2087
1      779
Name: count, dtype: int64

After balancing:
Severity
2    62111
3    62111
4    62111
1    62111
Name: count, dtype: int64


### Output

Verified the balancing step worked correctly (all classes equalized to 62,111 rows each) 
and confirmed the true F1 score at full precision after catching a stale-result issue from 
an earlier run.

- MLP (balanced) — F1: 0.7353
- Confirms LightGBM (F1: 0.7601) remains the strongest model, consistent with the original 
  Version 4 analysis (LightGBM: 0.7595, MLP: 0.7429)

**Final model comparison (Version 5, full dataset):**

| Model | F1 Score |
|---|---|
| Logistic Regression | 0.5179 |
| Random Forest | 0.7273 |
| **LightGBM** | **0.7601** |
| MLP (balanced) | 0.7353 |

**Decision: LightGBM selected as the production model.**

**Decision:** LightGBM selected as the production model, based on consistent results 
across both Version 4 (original) and Version 5 (full-dataset, MLflow-tracked) runs.

In [13]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
experiment = client.get_experiment_by_name("accident-severity-models")
runs = client.search_runs(experiment.experiment_id, order_by=["metrics.weighted_f1 DESC"])

for r in runs:
    print(r.data.tags.get("mlflow.runName"), "-", r.data.metrics.get("weighted_f1"))

lightgbm - 0.7600743301652221
mlp_balanced - 0.7353289099022717
random_forest - 0.7272808432970514
logistic_regression - 0.5178833534642983


In [14]:
# Get the LightGBM run specifically
lgbm_run = [r for r in runs if r.data.tags.get("mlflow.runName") == "lightgbm"][0]
run_id = lgbm_run.info.run_id
print(f"LightGBM run ID: {run_id}")

# Register it as a named model in the registry
model_uri = f"runs:/{run_id}/model"
registered_model = mlflow.register_model(model_uri, "accident-severity-lightgbm")
print(f"Registered as: {registered_model.name}, version {registered_model.version}")


LightGBM run ID: 9398d6f5eeee4908bac3e08367547818


Successfully registered model 'accident-severity-lightgbm'.
2026/06/23 06:16:37 WARNING mlflow.tracking._model_registry.fluent: Run with id 9398d6f5eeee4908bac3e08367547818 has no artifacts at artifact path 'model', registering model based on models:/m-d5adf41e3940446e8216c629f2548576 instead


Registered as: accident-severity-lightgbm, version 1


Created version '1' of model 'accident-severity-lightgbm'.


In [15]:
client.set_registered_model_alias(
    name="accident-severity-lightgbm",
    alias="production",
    version=1
)
print("LightGBM version 1 tagged with alias 'production'.")

LightGBM promoted to Production stage.


/tmp/ipykernel_58/1336825052.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


### Output

Registered LightGBM (the confirmed champion model) and promoted it using MLflow's current 
alias-based system (`production` alias) rather than the deprecated stage-based approach.